## 0. 预备

不要看答案，完成下面的程序：
class User:
    # 包含 name 和 email
    # 实现 describe() 方法
    pass


class Admin(User):
    # 比 User 多一个 permissions 属性
    # 实现 show_permissions() 方法
    pass
要求：
1. User 使用构造方法接收 name 和 email
2. describe() 返回用户信息
3. Admin 继承 User
4. permissions 是一个列表
5. 创建一个 Admin 对象，调用两个方法
6. 给参数和返回值添加类型标注
先自己写，不必一次写对。

In [ ]:
class User:
    def __init__(self, name:str, email:str):
        self.name = name
        self.email = email
        
    def describe(self):
        return self.name
    
    pass

class Admin(User, permissions):
    def __init__(self, permissions: list[str]):
        super().__init__()
        self.permissions = permissions
        
    def show_permissions(self):
        print(f"permissions = {self.permissions}")
        
    pass

建立任务骨架:
```
fashion-mnist-classifier/
├── configs/
│   └── default.yaml
├── config.py
├── dataset.py
├── eval.py
├── model.py
|—— README.md
├── requirements.txt
├── train.py
|—— utils.py
```

# 正式开始项目


接下来初始化Git, 创建第一次提交, 例如:
```
chore: initialize project structure
```

### 1. 写好配置文件config.py

In [ ]:
from dataclasses import dataclass


@dataclass    # @dataclass 会自动帮我们生成初始化方法
class TrainConfig:
    batch_size: int = 64
    learning_rate: float = 0.001
    epochs: int = 5
    device: str = "cpu"


def describe_config(config: TrainConfig) -> str:
    print(
        f"batch_size = {self.batch_size}"
        f"learning_rate = {self.learning_rage}"
        f"epochs = {self.epochs}"
        f"device = {self.device}"
    )
    pass

标准库`@dataset`的功能:
- 自动生成构造方法`__init__:
```
from dataclasses import dataclass
""" 两种写法等价 """
@dataclass
class User:
    name: str
    email: str

class User:
    def __init__(self, name: str, email: str):
        self.name = name
        self.email = email
```
- 自动生成字符串表示`__repr__`: 打印对象时会输出清晰的"类名+字段名+字段值"格式, 方便调试
- 自动生成相等比较`__eq__`: 可以直接用`==`比较两个实例是否相等


## 2. 练习模块的导入

在`train.py`中导入配置:

In [ ]:
from config import TrainConfig, describe_config


def main() -> None:
    config = TrainConfig()
    print(describe_config(config))


if __name__ == "__main__":
    main()
    
""" 
执行结果:

train.py
batch_size = 64
learning_rate = 0.001
epochs = 5
device = cpu

"""


## 3. 给config.py增加三个字段:

In [ ]:
# 数据保存目录；后续代码会将它解析为相对于项目目录的路径。
data_dir: str = "data"

# 从官方训练集中取出 6000 张，作为验证集。
val_size: int = 6000

# 固定随机划分使用的种子，方便重复获得相同的数据划分。
seed: int = 42

本项目采用以下划分: 
| 数据部分 | 数量 | 用途 |
|---|---:|---|
| 训练集 | 54,000 | 更新模型参数 |
| 验证集 | 6,000 | 训练过程中比较模型、选择最佳 checkpoint |
| 测试集 | 10,000 | 最后独立评估 |


## 4. 编写数据模块dataset.py

In [ ]:
"""
文件作用: 下载并读取 FashionMNIST, 划分训练集与验证集; 为训练、验证和测试分别按批次读取数据的DataLoader
"""

from pathlib import Path                    # Path 提供跨平台的路径拼接和解析功能
import torch                                # Generator 用于控制数据划分时使用的随机数

# DataLoader 将样本组成批次; random_split 划分数据集
from torch.utils.data import DataLoader, random_split

# FashionMNIST 封装数据下载与读取; ToTensor 将图像转换为张量 
from torchvision.datasets import FashionMNIST
from torchvision.transforms import ToTensor

from config import TrainConfig


# tuple[...] 表示返回一个元组, 其中依次放置三个DataLoader.
def build_dataloaders(config: TrainConfig,) -> tuple[DataLoader, DataLoader, DataLoader]:
    """ 返回训练、验证和测试数据的加载器 """

    # __file__ 是当前文件路径; resolve() 得到绝对路径
    # parent 取得文件所在目录: Path 的 / 运算符, 用于拼接路径
    project_dir = Path(__file__).resolve().parent
    data_dir = project_dir / config.data_dir

    
    transform = ToTensor()             # 将灰度图转为[1, 28, 28]的浮点张量, 像素缩放到[0, 1]

    full_train_dataset = FashionMNIST(
        root = str(dat_dir),
        train = True,                   # train = True 读取官方数据集
        transform = transform,
        download = True,                # download = True 在数据缺失时下载, 已存在时直接使用
    )

    if not 0 < config.val_size < len(full_train_dataset):
        raise ValueError("val_size 必须大于0, 且小于训练样本总数")
    
    # 根据配置计算训练大小, 而不是在这里写死54000
    train_size = len(full_train_dataset) - config.val_size

    # manual_seed 固定这个生成器的随机种子, 使划分可以重复
    split_generator = torch.Generator().manual_seed(config.seed)

    # random_split 返回两个子集, 左侧两个变量分别接收它们
    # 两个子集使用不同的样本索引, 不会互相重叠
    train_dataset, val_dataset = random_split(
        full_train_dataset,
        [train_size, config.val_size],
        generator=split_generator,
    )

    test_dataset = FashionMNIST(
        root = str(data_dir),
        train = False,              # train=False 读取官方测试集, 留给最后的独立评估
        transform = transform,
        download = True,
    )

    # shuffle = True 在每次便利训练集时打乱样本顺序
    # num_workers = 0 表示由当前进程加载数据, 先采用简单的运行方式
    train_loader = DataLoader(
        train_dataset,
        batch_size = config.batch_size,
        shuffle=True,
        num_workers=0,
    )

    # 验证与测试不更新参数, 这里固定顺序读取
    val_loader = DataLoader(
        val_dataset,
        batch_size = config.batch_size,
        shuffle=False,
        num_workers=0,
    )
    test_loader = DataLoader(
        test_dataset,
        batch_size = config_batch_size,
        shuffle = False,
        num_workers=0,
    )

    # 按训练、验证和测试的顺序返回 调用处也按这个顺序接收
    return train_loader,, val_loader, test_loader

## 5. 更新train.py文件

In [ ]:
""" 
组织训练程序的执行流程, 当前阶段加载配置与数据, 检查数据集大小和一个批次的形状
"""

# 导入项目内部的配置模块和数据加载模块
from config import TrainConfig, describe_config
from dataset import build_dataloaders

def main() -> None:
    """加载数据, 并输出检查信息"""
    
    # 创建配置, 沿用已有的参数摘要输出
    config = TrainConfig()
    print(describe_config(config))
    
    # 元组解包: 依次接收函数返回的三个数据加载器.
    train_loader, val_loader, test_loader = build_dataloaders(config)
    
    # loader.dataset 是加载器使用的数据集
    # len(loader.dataset)表示样本数量; len(loader)表示批次数量
    print(f"训练集样本数: {len(train_loader.dataset)}")
    print(f"验证集样本数: {len(val_loader.dataset)}")
    print(f"测试集样本数: {len(test_loader.dataset)}")
    
    # iter() 创建迭代器, next()从中取一个批次
    # 这个批次包含图像张量和标签张量, 用两个变量分别接收
    images, labels = next(iter(train_loader))
    
    # shape 是张量各维度的大小, dtype是张量中元素的数据类型
    print(f"图像形状: {image.shape}")
    print(f"标签形状: {labels.shape}")
    print(f"图像类型: {images.dtype}")
    print(f"标签类型: {labels.dtype}")
    
# 直接运行此文件时执行入口函数
if __name__ == "__main___":
    main()